# 🎓 LouisFarm Data Analytics Academy
## Semaine 1 — Data Analyst Foundations
### Contexte : Afrique de l'Ouest · Python · Pandas · NumPy

---
**Objectif :** Construire les fondamentaux du raisonnement analytique, maîtriser Python/Pandas et réaliser un premier diagnostic de données réelles ouest-africaines.

> **Note API :** Dans votre environnement local, vous pouvez utiliser la World Bank API :
> `https://api.worldbank.org/v2/country/TG;BJ;CI;GH/indicator/AG.YLD.CREL.KG?format=json`


In [ ]:
# ─── LEÇON 1.1 : IMPORTS ET CONFIGURATION ───────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
matplotlib.rcParams['font.family'] = 'DejaVu Sans'
import warnings; warnings.filterwarnings('ignore')

print("✅ Bibliothèques importées")
print(f"NumPy   : {np.__version__}")
print(f"Pandas  : {pd.__version__}")


---
## Leçon 1.1 : Rôle du Data Analyst en Afrique de l'Ouest

Un **Data Analyst** transforme les données brutes en informations décisionnelles.

**Exemples de questions analytiques en contexte ouest-africain :**
- Quelles régions du Togo ont les meilleurs rendements agricoles ?
- Quel est le profil type des utilisateurs de mobile money au Sénégal ?
- Les prix de l'immobilier à Abidjan ont-ils augmenté depuis 2020 ?

**Framework analytique :**
```
Contexte → Question → Données → Méthode → Résultat → Action
```


In [ ]:
# ─── LEÇON 1.2 : NUMPY FONDAMENTAUX ────────────────────────────────────────
print("=" * 55)
print("LEÇON 1.2 — NumPy : Calcul scientifique vectorisé")
print("=" * 55)

# Production agricole simulée (tonnes) - 5 régions, 4 ans
production = np.array([
    [12500, 13200, 11800, 14100],   # Maritime
    [18300, 17900, 19200, 20100],   # Plateaux
    [9800,  10200, 9500,  11000],   # Centrale
    [15600, 16100, 15200, 17300],   # Kara
    [7200,  7800,  7000,  8200],    # Savanes
])
regions = ['Maritime', 'Plateaux', 'Centrale', 'Kara', 'Savanes']
annees  = [2020, 2021, 2022, 2023]

print(f"Shape du tableau : {production.shape}  → ({len(regions)} régions, {len(annees)} années)")
print(f"\nProduction totale par région (tonnes):")
for r, total in zip(regions, production.sum(axis=1)):
    print(f"  {r:<12} : {total:>8,.0f} t")

print(f"\nStatistiques globales :")
print(f"  Moyenne  : {production.mean():>10,.0f} t")
print(f"  Max      : {production.max():>10,.0f} t  (région: {regions[production.max(axis=1).argmax()]})")
print(f"  Min      : {production.min():>10,.0f} t  (région: {regions[production.min(axis=1).argmin()]})")
print(f"  Croissance 2020→2023 : {((production[:,3]-production[:,0])/production[:,0]*100).mean():.1f}%")


In [ ]:
# ─── LEÇON 1.3 : PANDAS — PREMIER CONTACT AVEC DONNÉES RÉELLES ─────────────
print("=" * 55)
print("LEÇON 1.3 — Pandas & données agricoles Togo")
print("=" * 55)

import sys, os
sys.path.insert(0, ".")
from utils_louisfarm import gen_agriculture_togo

df = gen_agriculture_togo(n=1200)
print(f"\n📊 Dataset chargé : {df.shape[0]} observations × {df.shape[1]} variables")
print("\n🔍 Aperçu des 5 premières lignes :")
print(df.head())


In [ ]:
# ─── INSPECTION COMPLÈTE DU DATASET ─────────────────────────────────────────
print("\n📋 STRUCTURE DU DATASET")
print("-" * 50)
df.info()

print("\n📊 STATISTIQUES DESCRIPTIVES")
print("-" * 50)
print(df.describe().round(2))


In [ ]:
# ─── SÉLECTION ET FILTRAGE ───────────────────────────────────────────────────
print("\n🔎 SÉLECTION DE COLONNES")
print(df[['region', 'rendement_tonne_ha', 'type_semence']].head(8))

print("\n🔎 FILTRAGE : Semences améliorées en région Maritime")
mask = (df['type_semence'] == 'Améliorée') & (df['region'] == 'Maritime')
filtered = df[mask]
print(f"Nombre de records : {len(filtered)}")
print(f"Rendement moyen   : {filtered['rendement_tonne_ha'].mean():.2f} t/ha")

print("\n🔎 TRIAGE PAR RENDEMENT (Top 10)")
print(df.nlargest(10, 'rendement_tonne_ha')[['region','type_semence','rendement_tonne_ha','annee']])


In [ ]:
# ─── PREMIÈRES STATISTIQUES PAR GROUPE ────────────────────────────────────
print("\n📊 RENDEMENT MOYEN PAR RÉGION (t/ha)")
print("-" * 40)
by_region = df.groupby('region')['rendement_tonne_ha'].agg(['mean','std','count'])
by_region.columns = ['Moyenne','Écart-type','N']
by_region = by_region.sort_values('Moyenne', ascending=False)
print(by_region.round(2))

print("\n📊 RENDEMENT PAR TYPE DE SEMENCE")
print("-" * 40)
by_semence = df.groupby('type_semence')['rendement_tonne_ha'].agg(['mean','std','count'])
by_semence.columns = ['Moyenne','Écart-type','N']
print(by_semence.sort_values('Moyenne', ascending=False).round(2))


In [ ]:
# ─── LEÇON 1.4 : VISUALISATION BASIQUE ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle("Semaine 1 — Premiers graphiques : Agriculture Togo", 
             fontsize=14, fontweight='bold', y=1.02)

# Bar chart — rendement par région
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D', '#3B1F2B']
by_reg = df.groupby('region')['rendement_tonne_ha'].mean().sort_values()
axes[0].barh(by_reg.index, by_reg.values, color=colors)
axes[0].set_xlabel('Rendement moyen (t/ha)')
axes[0].set_title('Rendement moyen par région')
for i, v in enumerate(by_reg.values):
    axes[0].text(v+0.02, i, f'{v:.2f}', va='center', fontsize=10)

# Evolution temporelle
yearly = df.groupby('annee')['rendement_tonne_ha'].mean()
axes[1].plot(yearly.index, yearly.values, 'o-', color='#2E86AB', linewidth=2.5, markersize=8)
axes[1].fill_between(yearly.index, yearly.values, alpha=0.15, color='#2E86AB')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Rendement moyen (t/ha)')
axes[1].set_title('Évolution du rendement 2019-2023')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('./s1_viz.png', bbox_inches='tight', dpi=120)
plt.show()
print("Graphiques sauvegardés ✅")


---
## 🧪 EXERCICES — Semaine 1

### Exercice 1.1 — NumPy (★☆☆)
À partir du tableau `production` ci-dessous, calculez :
1. Le total de production par année (sum sur l'axe des régions)
2. L'année avec la plus forte production totale
3. Le pourcentage de croissance de chaque région entre 2020 et 2023


In [ ]:
# EXERCICE 1.1 — Complétez le code
production = np.array([
    [12500, 13200, 11800, 14100],
    [18300, 17900, 19200, 20100],
    [9800,  10200, 9500,  11000],
    [15600, 16100, 15200, 17300],
    [7200,  7800,  7000,  8200],
])
regions = ['Maritime', 'Plateaux', 'Centrale', 'Kara', 'Savanes']
annees  = [2020, 2021, 2022, 2023]

# 1. Total par année
total_par_annee = production.sum(axis=0)   # ← axis=0 = somme des lignes (régions)
print("Total par année:", dict(zip(annees, total_par_annee)))

# 2. Année avec la plus forte production
idx_max = total_par_annee.argmax()
print(f"Meilleure année : {annees[idx_max]} ({total_par_annee[idx_max]:,.0f} t)")

# 3. Croissance par région
croissance = (production[:,3] - production[:,0]) / production[:,0] * 100
for r, c in zip(regions, croissance):
    print(f"  {r:<12}: {c:+.1f}%")


### Exercice 1.2 — Pandas (★★☆)
Avec le dataset `df` (agriculture Togo) :
1. Combien de villages différents sont présents dans le dataset ?
2. Quel type de semence génère le meilleur prix de vente moyen ?
3. Filtrez les observations avec rendement > 2.5 t/ha ET accès à l'irrigation. Combien y en a-t-il ?


In [ ]:
# EXERCICE 1.2 — Solutions
# 1. Villages uniques
print(f"1. Nombre de villages : {df['village'].nunique()}")
print(f"   Villages : {df['village'].unique().tolist()}")

# 2. Prix de vente par type de semence
print("\n2. Prix moyen par type de semence (FCFA) :")
prix_semence = df.groupby('type_semence')['prix_vente_fcfa'].mean().sort_values(ascending=False)
for s, p in prix_semence.items():
    print(f"   {s:<15}: {p:>12,.0f} FCFA")

# 3. Filtrage
haut_rend = df[(df['rendement_tonne_ha'] > 2.5) & (df['acces_irrigation'] == 1)]
print(f"\n3. Observations (rendement>2.5 AND irrigation) : {len(haut_rend)}")
print(f"   Soit {len(haut_rend)/len(df)*100:.1f}% du dataset")


---
## 📋 LAB GUIDÉ — Diagnostic d'un dataset de coopérative

**Scénario :** La Coopérative Agro-Togo vous envoie ses données. Votre mission : produire un rapport de diagnostic en 15 minutes.


In [ ]:
# LAB GUIDÉ 1 — Diagnostic complet en 7 étapes

print("╔══════════════════════════════════════════════════╗")
print("║   DIAGNOSTIC — Coopérative Agro-Togo            ║")
print("╚══════════════════════════════════════════════════╝\n")

# ÉTAPE 1 : Chargement
df_lab = gen_agriculture_togo(n=800, seed=123)
print(f"ÉTAPE 1 — Chargement : {df_lab.shape[0]} lignes × {df_lab.shape[1]} colonnes\n")

# ÉTAPE 2 : Structure
print("ÉTAPE 2 — Types de variables :")
type_map = {'object':'Catégorielle','int64':'Entier','float64':'Numérique continue'}
for col in df_lab.columns:
    dtype = str(df_lab[col].dtype)
    mapped = type_map.get(dtype, dtype)
    print(f"  {col:<25} → {mapped}")

# ÉTAPE 3 : Valeurs manquantes
print("\nÉTAPE 3 — Valeurs manquantes :")
missing = df_lab.isnull().sum()
for col, n_miss in missing[missing>0].items():
    pct = n_miss/len(df_lab)*100
    print(f"  {col:<25} : {n_miss:>4} ({pct:.1f}%)")

# ÉTAPE 4 : Doublons
print(f"\nÉTAPE 4 — Doublons : {df_lab.duplicated().sum()} lignes")

# ÉTAPE 5 : Statistiques clés
print("\nÉTAPE 5 — Variable cible : rendement_tonne_ha")
rend = df_lab['rendement_tonne_ha']
print(f"  Moyenne  : {rend.mean():.3f} t/ha")
print(f"  Médiane  : {rend.median():.3f} t/ha")
print(f"  Min-Max  : {rend.min():.2f} — {rend.max():.2f} t/ha")

# ÉTAPE 6 : Distribution par région
print("\nÉTAPE 6 — Observations par région :")
print(df_lab['region'].value_counts().to_string())

# ÉTAPE 7 : Questions analytiques
print("\nÉTAPE 7 — 3 questions analytiques identifiées :")
print("  Q1 : Quel est l'impact de l'irrigation sur le rendement par région ?")
print("  Q2 : La pluviométrie explique-t-elle les variations de rendement ?")
print("  Q3 : Quelle combinaison (région + semence) maximise le rendement ?")

print("\n✅ Diagnostic terminé — Prêt pour la Semaine 2")


---
## 🎯 CRT — Critères de Validation Semaine 1

Pour débloquer la Semaine 2, vous devez :
- [ ] Répondre au quiz CRT sur louisfarm.com (score ≥ 70%)
- [ ] Soumettre votre notebook projet avec 3 questions analytiques rédigées
- [ ] Obtenir la validation de votre diagnostic

**Rappel :** Le score minimum est de **70%** sur le CRT.


In [ ]:
# ─── API RÉELLE (À UTILISER DANS VOTRE ENVIRONNEMENT LOCAL) ──────────────────
# Décommentez ce code dans votre environnement local (pas de restriction réseau)

# import requests, json

# # World Bank API — Production céréalière en Afrique de l'Ouest
# url = "https://api.worldbank.org/v2/country/TG;BJ;CI;GH;SN/indicator/AG.YLD.CREL.KG"
# params = {"format": "json", "date": "2015:2023", "per_page": 100}
# response = requests.get(url, params=params)
# data = response.json()

# records = []
# for item in data[1]:
#     if item['value']:
#         records.append({
#             'pays': item['country']['value'],
#             'code': item['countryiso3code'],
#             'annee': int(item['date']),
#             'rendement_kg_ha': item['value']
#         })

# df_wb = pd.DataFrame(records)
# print(df_wb.head(15))

print("ℹ️  Pour utiliser la World Bank API dans votre environnement :")
print("   pip install requests")
print("   URL : https://api.worldbank.org/v2/country/TG;BJ;CI;GH/indicator/AG.YLD.CREL.KG?format=json")
print()
print("ℹ️  Pour l'API Open-Meteo (météo temps réel, GRATUITE) :")
print("   URL : https://api.open-meteo.com/v1/forecast?latitude=6.14&longitude=1.22")
print("         &hourly=temperature_2m,precipitation&past_days=30")
